# Notebook 05 — Results Writing

**Module:** 18 — Scientific Writing and Open Science  
**Tier:** 2 — Working competence  
**Notebook:** 5 of 12  
**Time estimate:** 60 minutes

> Results is the only section of a paper where your data speaks.
> Every sentence must be a claim supported by a figure or a table.
> Interpretation belongs in Discussion — not here.

---
## Step 1 — Motivation

The Results section is the most frequently mined part of a paper in literature
surveys and meta-analyses. Readers who skim go Abstract → Figures → Results text.
If the Results text doesn't match the figure, or the quantitative claims are vague,
the paper fails at the most critical juncture.

---
## Step 2 — Figure-First Principle

**Design the figures first. Write the results text second.**

Each figure panel answers one question. The results text for that panel:
1. States the question the panel answers (1 sentence)
2. References the panel: "(Figure 2A)" — before the claim, not after
3. States the quantitative result: the number, the comparison, and the uncertainty
4. Notes any exception or condition (if the result doesn't hold everywhere, say so)

**Never:** describe a figure without a number. Never write "the figure shows..." —
write "(Figure 2A) DeepBind-v2 achieves...".

---
## Step 3 — Statistical Reporting Standards

| What to report | Format example |
|---------------|---------------|
| Point estimate + uncertainty | mean AUROC 0.924 ± 0.008 (SD) |
| Confidence interval | AUROC 0.924 (95% CI: 0.908–0.940) |
| Comparison with test | outperformed PWM baseline by 12.3 pp (Wilcoxon, p < 0.001) |
| Effect size | Cohen's d = 1.42 (large effect) |
| Multiple testing | Bonferroni-corrected p < 0.05 for 690 comparisons |
| N | n = 690 TFs, 5 independent replicates per TF |

**Never use "significant" without a p-value.** "Significantly better" is meaningless.
"Better by 12.3 percentage points (Wilcoxon signed-rank, p < 0.001, n = 690)" is a claim.

**Effect size > p-value.** A p < 0.001 with a 0.1% effect size in N=10,000 is not
a meaningful result. Always report the effect size alongside the test.

---
## Step 4 — Quantitative Precision Rules

| Situation | Rule |
|-----------|------|
| Proportions / accuracy | 1–2 decimal places (0.92, not 0.9241837) |
| p-values | Report exactly if p ≥ 0.001; use "p < 0.001" below that |
| Effect size | 2 decimal places |
| Sample sizes | Integer, with no trailing zeros |
| Fold changes | 1 decimal place (3.2-fold, not 3.18734-fold) |

**Overclaiming:** "AUROC of 0.9241 demonstrates our method's superiority" —
that fourth decimal is not meaningful. Round to 0.924.
**Underclaiming:** "results showed improvement" — how much? over what baseline?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re
from scipy import stats

rng = np.random.default_rng(42)

# ---- Synthetic experiment data (mimicking a TF binding study) ----

n_tfs = 50
auroc_deepbind = rng.beta(9, 1, n_tfs)       # mean ~0.90
auroc_pwm      = rng.beta(7, 2, n_tfs)       # mean ~0.78

# Paired comparison
diff = auroc_deepbind - auroc_pwm
stat, pvalue = stats.wilcoxon(diff, alternative='greater')

print('=== Synthetic experiment: DeepBind-v2 vs PWM baseline ===')
print(f'DeepBind-v2:  mean AUROC = {auroc_deepbind.mean():.3f} ± {auroc_deepbind.std():.3f}')
print(f'PWM baseline: mean AUROC = {auroc_pwm.mean():.3f} ± {auroc_pwm.std():.3f}')
print(f'Mean improvement: {diff.mean()*100:.1f} pp')
print(f'Wilcoxon signed-rank: W = {stat:.1f}, p = {pvalue:.4f}')
print(f'TFs improved: {(diff>0).sum()}/{n_tfs}')

# Compute 95% CI for mean difference via bootstrapping
n_boot = 5000
boot_means = [rng.choice(diff, len(diff), replace=True).mean() for _ in range(n_boot)]
ci_lo, ci_hi = np.percentile(boot_means, [2.5, 97.5])
print(f'95% CI for mean diff: ({ci_lo*100:.1f}, {ci_hi*100:.1f}) pp')

# ---- Results text generation ----
def format_results_sentence(method_name, baseline_name, mean_method, std_method,
                              mean_base, std_base, diff_pp, ci_lo_pp, ci_hi_pp,
                              pvalue, n, figure_ref):
    """
    Generate a properly formatted results sentence following the reporting standard.
    """
    p_str = f'p < 0.001' if pvalue < 0.001 else f'p = {pvalue:.3f}'
    return (
        f'{figure_ref} {method_name} achieves a mean AUROC of '
        f'{mean_method:.3f} ± {std_method:.3f} (SD), compared to '
        f'{mean_base:.3f} ± {std_base:.3f} for the {baseline_name} baseline. '
        f'The mean improvement of {diff_pp:.1f} percentage points '
        f'(95% CI: {ci_lo_pp:.1f}–{ci_hi_pp:.1f} pp) is significant '
        f'(Wilcoxon signed-rank, {p_str}, n = {n} TFs).'
    )

result_text = format_results_sentence(
    'DeepBind-v2', 'PWM',
    auroc_deepbind.mean(), auroc_deepbind.std(),
    auroc_pwm.mean(),      auroc_pwm.std(),
    diff.mean()*100, ci_lo*100, ci_hi*100,
    pvalue, n_tfs,
    '(Figure 2A)'
)
print(f'\n=== Formatted results sentence ===\n{result_text}')

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel A: Paired AUROC comparison (strip plot)
ax = axes[0]
jitter = rng.uniform(-0.1, 0.1, n_tfs)
for i in range(n_tfs):
    ax.plot([0 + jitter[i], 1 + jitter[i]], [auroc_pwm[i], auroc_deepbind[i]],
              color='steelblue', alpha=0.25, lw=0.7, zorder=1)
ax.scatter(np.zeros(n_tfs) + jitter, auroc_pwm,      color='tomato',    s=15, zorder=2, label='PWM baseline')
ax.scatter(np.ones(n_tfs)  + jitter, auroc_deepbind, color='steelblue', s=15, zorder=2, label='DeepBind-v2')
for x_pos, vals, color in [(0, auroc_pwm, 'tomato'), (1, auroc_deepbind, 'steelblue')]:
    ax.errorbar(x_pos, vals.mean(), yerr=vals.std(), fmt='D', color=color,
                  markersize=8, capsize=5, lw=2, zorder=3)
ax.set_xticks([0, 1]); ax.set_xticklabels(['PWM\nbaseline', 'DeepBind-v2'])
ax.set_ylabel('AUROC'); ax.set_ylim(0.4, 1.02)
ax.set_title(f'A. Paired AUROC comparison\n(n={n_tfs} TFs; p < 0.001)')
ax.legend(fontsize=8)

# Panel B: Distribution of improvements
ax = axes[1]
ax.hist(diff * 100, bins=15, color='steelblue', edgecolor='black', alpha=0.8)
ax.axvline(0, color='black', ls='-', lw=1.5, label='No change')
ax.axvline(diff.mean()*100, color='tomato', ls='--', lw=2,
             label=f'Mean = {diff.mean()*100:.1f} pp')
ax.fill_betweenx([0, 18], ci_lo*100, ci_hi*100, alpha=0.2, color='tomato', label='95% CI')
ax.set_xlabel('AUROC improvement (percentage points)')
ax.set_ylabel('Number of TFs')
ax.set_title('B. Distribution of per-TF improvement\n(DeepBind-v2 − PWM baseline)')
ax.legend(fontsize=8)

# Panel C: Statistical reporting checklist
ax = axes[2]
ax.axis('off')
checklist = [
    ('Point estimate reported?',         True),
    ('Uncertainty (SD or SE) given?',    True),
    ('Comparison to baseline included?', True),
    ('Test statistic named?',            True),
    ('p-value exact or < threshold?',    True),
    ('Sample size stated?',              True),
    ('Effect size reported?',            True),  # pp improvement
    ('95% CI provided?',                 True),
    ('Figure cited before claim?',       True),
    ('"Significant" without p-value?',   False),  # False = avoided this mistake
]
for i, (item, good) in enumerate(checklist):
    y = 0.95 - i * 0.09
    symbol = '✓' if good else '✗'
    color  = '#2ca02c' if good else '#d62728'
    ax.text(0.04, y, symbol, transform=ax.transAxes, fontsize=12,
              color=color, va='center', fontweight='bold')
    ax.text(0.12, y, item, transform=ax.transAxes, fontsize=8.5, va='center')
ax.set_title('C. Statistical reporting checklist\n(for the results sentence above)')

plt.suptitle('Module 18 NB05: Results Writing', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('results_writing.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

See E05 in `exercises/README.md`: given three vague results sentences, rewrite
each to meet the statistical reporting standard above.

---
## Step 10 — Quiz

1. What is the figure-first principle? Why does it matter for writing?
2. Rewrite: "Our method showed significant improvement over the baseline."
   Use: AUROC 0.91 vs 0.83, n=200, paired t-test p=0.003, Cohen's d=0.84.
3. What is the difference between standard deviation and standard error of the mean?
   Which should you report in a bar chart error bar for a biological experiment?
4. When should you report "p < 0.001" versus "p = 0.0003"?

---
## Step 12 — Reflection

> *[For the result you chose for MP01, write one results paragraph: one sentence
> referencing the figure panel, one sentence with the quantitative result in full
> reporting format, one sentence noting any exception or limitation of the result.
> Then run a passive-voice check on your paragraph.]*

---
*Next: `06_discussion_and_conclusions.ipynb`*